# T3 — Mã hoá BGE-M3 toàn kho (3 người, mỗi người một phần)

**Cài đặt notebook:** Accelerator = **GPU T4 ×2**, Internet = **On** (tải `BAAI/bge-m3`).
**Add Input:** `mir-code`, `mir-data` (bản có `corpus_clean/` và `chunks/structure/`).
Ba người phải dùng **cùng một phiên bản** `mir-data`: dòng `mir-data … dấu vân tay …` in ở ô (1) phải giống nhau.
Nếu lệch, bước gộp ở T5 sẽ từ chối các khối.

**Chia việc:** 3 người × 2 GPU = 6 phần. Người `m` đặt `MEMBER = m` (0, 1 hoặc 2). GPU 0 chạy phần `2m/6`, GPU 1 chạy
phần `2m+1/6`. Mỗi khối 100.000 đoạn ghi vào `emb/structure/shard-XXXX/` kèm `_DONE.json`.

**Cách chạy:** đặt `MEMBER` → *Save Version* → *Save & Run All (Commit)*. Notebook chạy nền, không cần mở trình
duyệt. Ước tính khoảng 4,3–5 giờ mỗi người, từ tốc độ 185 đoạn/s đo ngày 05/10 với kho ước lượng 8,5 triệu đoạn
(chưa đo trên kho thật). Xong thì vào version vừa chạy → tab *Output* → *New Dataset*, đặt tên **`mir-emb-m{MEMBER}`**
và chia sẻ cho người chạy T5.

**Bị ngắt giữa chừng:** Add Input thêm output của version bị ngắt rồi chạy lại. Ô (3) chép các khối đã xong của
`MEMBER` rồi mã hoá tiếp phần còn lại.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây; phần còn lại giống nhau ở mọi notebook.
NB_VERSION = 't3-v2 (05/10/2026)'
MEMBER = None               # ← ĐẶT 0, 1 hoặc 2 (mỗi người một số khác nhau)
RUN_BENCH = False           # True: đo tốc độ trên 2.000 đoạn mẫu trước khi chạy (khoảng 1 phút)
OVERRIDES = {}

import glob, os, shutil, sys
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')          # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mir/__init__.py', recursive=True), key=len)
assert hits, 'Chưa thêm dataset mir-code (Add Input → Your Datasets)'
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)                     # luôn chép lại bản mir-code mới nhất
shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mir.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, OVERRIDES, corpus='full')
assert MEMBER in (0, 1, 2), 'Đặt MEMBER = 0, 1 hoặc 2 ở đầu ô này'
import json

In [ ]:
# (2) Đo tốc độ (tuỳ chọn) trên kho mẫu, 2 GPU cùng lúc
if RUN_BENCH:
    cmd = (f'python -m mir.encode --config {K.cfg} --bench 2000 --est-chunks 8500000 '
           f'--input {K.data}/sample/chunks/structure --clean {K.data}/sample/corpus_clean')
    par([f'{cmd} --device cuda:{g}' for g in range(2)], 'bench')
    for g in range(2):
        print(json.load(open(f'{K.work}/results/t3_bench_cuda{g}.json')))

In [ ]:
# (3) Chép các khối đã xong của MEMBER từ output lần chạy trước (nếu đã Add Input), rồi mã hoá phần của MEMBER trên 2 GPU
OUT = f'{K.work}/emb/structure'
def mine(k):                                # khối k thuộc phần của MEMBER
    return k % 6 in (2 * MEMBER, 2 * MEMBER + 1)
for d in locate_all('emb/structure'):
    for s in sorted(glob.glob(f'{d}/emb/structure/shard-*')):
        name = os.path.basename(s)
        if mine(int(name.split('-')[1])) and os.path.exists(f'{s}/_DONE.json') and not os.path.exists(f'{OUT}/{name}'):
            shutil.copytree(s, f'{OUT}/{name}')
            print('chép khối đã xong:', s)
cmd = f'python -m mir.encode --config {K.cfg}'
par([f'{cmd} --shard {2 * MEMBER + g}/6 --device cuda:{g}' for g in range(2)], f'T3_member{MEMBER}')

In [ ]:
# (4) Kiểm tra: mọi khối của MEMBER đã xong chưa → nếu đủ thì tạo dataset mir-emb-m{MEMBER} từ output
from mir import config, encode
c = config.load(K.cfg)
n_blocks = -(-sum(n for _, n in encode.chunk_parts(c.path('chunks', 'structure'))) // c['encode']['shard_size'])
todo = [k for k in range(n_blocks) if mine(k)]
missing = [k for k in todo if not os.path.exists(f'{OUT}/shard-{k:04d}/_DONE.json')]
print(f'MEMBER {MEMBER}: xong {len(todo) - len(missing)}/{len(todo)} khối' + (f'; CÒN THIẾU {missing}' if missing else ''))
report(f't3_m{MEMBER}')